# 04 — Modelado de Grafo Gold para Neo4j Desktop y GraphRAG
Genera las tablas dimensionales `gold.pdac.graph_nodes` y `gold.pdac.graph_relationships`, exporta CSV planos a un Volumen UC y valida el 100% de integridad referencial.

**1. Imports, tablas Gold y Volumen de exportación**
Configura los identificadores de tablas en `gold.pdac`, las prioridades de etiquetas y la ruta de exportación para Neo4j Desktop.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE = "silver.pdac.clean_triplets"
GOLD_NODES_TABLE = "gold.pdac.graph_nodes"
GOLD_RELS_TABLE = "gold.pdac.graph_relationships"
EXPORT_VOLUME_NAME = "gold.pdac.neo4j_export_vol"
EXPORT_BASE_PATH = "/Volumes/gold/pdac/neo4j_export_vol"

# Prioridad determinista ante cualquier conflicto residual de etiqueta de nodo
NODE_TYPE_PRIORITY = {
    "DRUG": 1,
    "REGIMEN": 2,
    "GENE": 3,
    "MUTATION": 4,
    "BIOMARKER": 5,
    "PATHWAY": 6,
    "DISEASE": 7,
    "STAGE": 8,
    "PROCEDURE": 9,
    "TRIAL": 10,
    "RISK_FACTOR": 11,
    "SYMPTOM": 12,
    "ADVERSE_EVENT": 13,
}

**2. Construcción de la tabla de nodos (`gold.pdac.graph_nodes`)**
Unifica sujetos y objetos de Silver, resuelve etiquetas mayoritarias, genera `node_id` determinista SHA-256 y consolida frecuencias y fuentes.

In [0]:
silver_df = spark.table(SILVER_TABLE)

subj_nodes = silver_df.select(
    F.col("subject").alias("name"),
    F.col("subject_type").alias("label"),
    F.col("evidence_count"),
    F.col("source_domains"),
    F.col("source_docs"),
)
obj_nodes = silver_df.select(
    F.col("object").alias("name"),
    F.col("object_type").alias("label"),
    F.col("evidence_count"),
    F.col("source_domains"),
    F.col("source_docs"),
)

all_appearances = subj_nodes.unionAll(obj_nodes)

# Selección determinista de la etiqueta principal (mayor mención y prioridad clínica)
prio_expr = F.create_map([F.lit(item) for kv in NODE_TYPE_PRIORITY.items() for item in kv])
label_scores = all_appearances.groupBy("name", "label").agg(
    F.sum("evidence_count").alias("label_mentions")
)
w_label = Window.partitionBy("name").orderBy(
    F.desc("label_mentions"),
    F.coalesce(F.element_at(prio_expr, F.col("label")), F.lit(99)),
    F.asc("label"),
)
authoritative_labels = (
    label_scores.withColumn("_rn", F.row_number().over(w_label))
    .filter(F.col("_rn") == 1)
    .select("name", "label")
)

node_agg = (
    all_appearances.groupBy("name")
    .agg(
        F.sum("evidence_count").cast("int").alias("mention_count"),
        F.count("*").cast("int").alias("degree"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list("source_domains")))).alias("_domains_arr"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list("source_docs")))).alias("_docs_arr"),
    )
    .withColumn("doc_count", F.size(F.col("_docs_arr")).cast("int"))
    .withColumn("sources", F.concat_ws(";", F.col("_domains_arr")))
    .withColumn("source_docs", F.concat_ws(";", F.col("_docs_arr")))
    .drop("_domains_arr", "_docs_arr")
)

graph_nodes_df = (
    node_agg.join(authoritative_labels, on="name", how="inner")
    .withColumn(
        "node_id",
        F.concat(F.lit("n_"), F.substring(F.sha2(F.lower(F.trim(F.col("name"))), 256), 1, 16)),
    )
    .select(
        "node_id",
        "name",
        "label",
        "mention_count",
        "degree",
        "doc_count",
        "sources",
        "source_docs",
    )
    .orderBy(F.desc("mention_count"), F.asc("name"))
)

(
    graph_nodes_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_NODES_TABLE)
)
spark.sql(f"COMMENT ON TABLE {GOLD_NODES_TABLE} IS 'Capa Gold: Nodos canónicos deduplicados listos para ingesta en Neo4j Desktop.'")
print(f"Tabla de nodos guardada: {GOLD_NODES_TABLE} ({graph_nodes_df.count()} nodos)")

**3. Construcción de la tabla de aristas (`gold.pdac.graph_relationships`)**
Mapea cada tripleta validada a `source_id` y `target_id` mediante joins con `graph_nodes` y genera `rel_id` único SHA-256.

In [0]:
nodes_base = spark.table(GOLD_NODES_TABLE)
src_nodes = F.broadcast(
    nodes_base.select(
        F.col("node_id").alias("_src_id"),
        F.col("name").alias("_src_name"),
        F.col("label").alias("_src_label"),
    )
)
tgt_nodes = F.broadcast(
    nodes_base.select(
        F.col("node_id").alias("_tgt_id"),
        F.col("name").alias("_tgt_name"),
        F.col("label").alias("_tgt_label"),
    )
)

edges_joined = (
    spark.table(SILVER_TABLE)
    .join(src_nodes, F.col("subject") == F.col("_src_name"), "inner")
    .join(tgt_nodes, F.col("object") == F.col("_tgt_name"), "inner")
)

graph_relationships_df = (
    edges_joined.select(
        F.concat(
            F.lit("r_"),
            F.substring(
                F.sha2(
                    F.concat_ws("|", F.col("_src_id"), F.col("predicate"), F.col("_tgt_id")),
                    256,
                ),
                1,
                16,
            ),
        ).alias("rel_id"),
        F.col("_src_id").alias("source_id"),
        F.col("_tgt_id").alias("target_id"),
        F.col("_src_name").alias("source_name"),
        F.col("_src_label").alias("source_label"),
        F.col("predicate").alias("relationship_type"),
        F.col("_tgt_name").alias("target_name"),
        F.col("_tgt_label").alias("target_label"),
        F.col("evidence_count").cast("int").alias("evidence_count"),
        F.col("avg_confidence").cast("double").alias("avg_confidence"),
        F.col("max_confidence").cast("double").alias("max_confidence"),
        F.col("doc_count").cast("int").alias("doc_count"),
        F.concat_ws(";", F.col("source_domains")).alias("source_domains"),
        F.concat_ws(";", F.col("source_files")).alias("source_files"),
    )
    .orderBy(F.desc("evidence_count"), F.desc("avg_confidence"), F.asc("rel_id"))
)

(
    graph_relationships_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_RELS_TABLE)
)
spark.sql(f"COMMENT ON TABLE {GOLD_RELS_TABLE} IS 'Capa Gold: Aristas relacionales con claves foráneas verificadas hacia graph_nodes para Neo4j Desktop.'")
print(f"Tabla de relaciones guardada: {GOLD_RELS_TABLE} ({graph_relationships_df.count()} aristas)")

**4. Exportación de archivos CSV planos a Volumen de Unity Catalog**
Crea el Volumen `gold.pdac.neo4j_export_vol` si no existe y exporta `graph_nodes` y `graph_relationships` en un único CSV cada uno para Neo4j Desktop.

In [0]:
spark.sql(
    f"""CREATE VOLUME IF NOT EXISTS {EXPORT_VOLUME_NAME}
    COMMENT 'Volumen de exportación de nodos y aristas en CSV para Neo4j Desktop'"""
)

nodes_export_path = f"{EXPORT_BASE_PATH}/graph_nodes"
rels_export_path = f"{EXPORT_BASE_PATH}/graph_relationships"

(
    spark.table(GOLD_NODES_TABLE)
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .option("escape", '"')
    .option("quoteAll", "true")
    .csv(nodes_export_path)
)

(
    spark.table(GOLD_RELS_TABLE)
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .option("escape", '"')
    .option("quoteAll", "true")
    .csv(rels_export_path)
)

print(f"CSV de nodos exportado en      : {nodes_export_path}")
print(f"CSV de relaciones exportado en : {rels_export_path}")

**5. Validación estricta de integridad referencial y métricas del grafo**
Comprueba cero aristas huérfanas, unicidad de claves primarias, ausencia de auto-bucles y muestra estadísticas de nodos y aristas.

In [0]:
gold_nodes = spark.table(GOLD_NODES_TABLE)
gold_rels = spark.table(GOLD_RELS_TABLE)

total_nodes = gold_nodes.count()
total_rels = gold_rels.count()

orphan_sources = gold_rels.join(gold_nodes, gold_rels["source_id"] == gold_nodes["node_id"], "left_anti").count()
orphan_targets = gold_rels.join(gold_nodes, gold_rels["target_id"] == gold_nodes["node_id"], "left_anti").count()
dup_node_ids = total_nodes - gold_nodes.select("node_id").distinct().count()
dup_node_names = total_nodes - gold_nodes.select("name").distinct().count()
dup_rel_ids = total_rels - gold_rels.select("rel_id").distinct().count()
self_loops = gold_rels.filter(F.col("source_id") == F.col("target_id")).count()
null_keys = (
    gold_nodes.filter(F.col("node_id").isNull() | F.col("name").isNull() | F.col("label").isNull()).count()
    + gold_rels.filter(
        F.col("rel_id").isNull()
        | F.col("source_id").isNull()
        | F.col("target_id").isNull()
        | F.col("relationship_type").isNull()
    ).count()
)

assert orphan_sources == 0, f"Detectadas {orphan_sources} aristas con source_id huérfano"
assert orphan_targets == 0, f"Detectadas {orphan_targets} aristas con target_id huérfano"
assert dup_node_ids == 0, f"Detectados {dup_node_ids} node_id duplicados"
assert dup_node_names == 0, f"Detectados {dup_node_names} nombres duplicados"
assert dup_rel_ids == 0, f"Detectados {dup_rel_ids} rel_id duplicados"
assert self_loops == 0, f"Detectados {self_loops} auto-bucles"
assert null_keys == 0, f"Detectados {null_keys} registros con claves nulas"

print("=== VERIFICACIÓN DE INTEGRIDAD REFERENCIAL: PASS (100%) ===")
print(f"Total Nodos Gold      : {total_nodes}")
print(f"Total Aristas Gold    : {total_rels}")
print(f"Aristas huérfanas     : {orphan_sources + orphan_targets}")
print(f"Claves duplicadas     : {dup_node_ids + dup_rel_ids}")

print("\n--- Nodos por etiqueta Neo4j ---")
gold_nodes.groupBy("label").count().orderBy(F.desc("count")).show(truncate=False)

print("--- Aristas por tipo de relación ---")
gold_rels.groupBy("relationship_type").agg(
    F.count("*").alias("edges"),
    F.sum("evidence_count").alias("total_evidence"),
).orderBy(F.desc("edges")).show(truncate=False)

print("--- Top 15 nodos centrales (Hubs) por grado ---")
gold_nodes.select("node_id", "name", "label", "degree", "mention_count", "doc_count").orderBy(
    F.desc("degree")
).show(15, truncate=False)

**6. Inspección de subgrafos clínicos reales de PDAC**
Muestra las conexiones reales alrededor de `KRAS`, `Gemcitabine`, `CA19-9` y `Pancreatic Ductal Adenocarcinoma`.

In [0]:
focus_entities = [
    "KRAS",
    "KRAS G12D",
    "Gemcitabine",
    "Nab-Paclitaxel",
    "CA19-9",
    "Pancreatic Ductal Adenocarcinoma",
]

for entity in focus_entities:
    print(f"\n=== Subgrafo: {entity} ===")
    subgraph_df = (
        gold_rels.filter((F.col("source_name") == entity) | (F.col("target_name") == entity))
        .select(
            "source_name",
            "source_label",
            "relationship_type",
            "target_name",
            "target_label",
            "evidence_count",
            "avg_confidence",
        )
        .orderBy(F.desc("evidence_count"), F.desc("avg_confidence"))
    )
    subgraph_df.show(12, truncate=False)